# Session 02 Practice: 2D LiDAR Occupancy Grid Mapping
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 1 | Session 2**  

---

### 🎯 Learning Objectives
1. Implement a **2D Log-Odds Occupancy Grid** mapping pipeline from first principles.
2. Build an **Inverse Sensor Model (ISM)** for range-bearing LiDAR beams.
3. Perform raycasting and log-odds updates ($l_{\text{occ}}, l_{\text{free}}$) across multiple vehicle poses.
4. Convert log-odds back into probabilistic occupancy $P(m_i = 1 | z_{1:t})$ and visualize spatial 2D heatmaps using **Plotly**.

---

### 📐 Mathematical Formulation

#### Binary Bayes Filter in Log-Odds Space
To avoid numerical underflow from multiplying small probabilities, we map probability $p \in (0, 1)$ into log-odds $l \in (-\infty, \infty)$:

$$l = \ln\left(\frac{p}{1 - p}\right) \iff p = \frac{1}{1 + e^{-l}}$$

For each LiDAR beam from sensor pose $(x_s, y_s)$ at angle $\theta_{\text{beam}}$ with measured range $r$:
1. **Free Cells along ray** ($d < r - \frac{\text{thickness}}{2}$):
   $$l_{t, i} = l_{t-1, i} + l_{\text{free}}$$
2. **Occupied Cell at hit point** ($|d - r| \le \frac{\text{thickness}}{2}$):
   $$l_{t, i} = l_{t-1, i} + l_{\text{occ}}$$
3. **Clamping:** $l \in [l_{\min}, l_{\max}]$ to prevent filter overconfidence.


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Occupancy Grid Data Structure & Inverse Sensor Model

In [ ]:
class OccupancyGrid2D:
    def __init__(self, x_min: float = -5.0, x_max: float = 25.0,
                 y_min: float = -15.0, y_max: float = 15.0,
                 resolution: float = 0.2):
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.resolution = resolution
        
        self.width = int((x_max - x_min) / resolution)
        self.height = int((y_max - y_min) / resolution)
        
        # Log-odds probability array initialized to 0 (P=0.5 prior)
        self.log_odds = np.zeros((self.height, self.width), dtype=np.float32)
        
        # Log-odds parameters
        self.l_occ = math.log(0.85 / (1.0 - 0.85))    # ~ +1.73
        self.l_free = math.log(0.30 / (1.0 - 0.30))   # ~ -0.85
        self.l_min = -4.0
        self.l_max = 4.0
        self.thickness = 0.3 # meters

    def world_to_grid(self, x: float, y: float) -> tuple:
        col = int((x - self.x_min) / self.resolution)
        row = int((y - self.y_min) / self.resolution)
        if 0 <= col < self.width and 0 <= row < self.height:
            return row, col
        return None

    def update_scan(self, robot_x: float, robot_y: float, robot_theta: float,
                    ranges: np.ndarray, angles: np.ndarray, max_range: float = 15.0):
        for angle, r in zip(angles, ranges):
            if r <= 0.1:
                continue
            
            beam_angle = robot_theta + angle
            is_valid_hit = (r < max_range)
            ray_len = min(r, max_range)
            
            # Step along raycast with half-cell resolution steps
            num_steps = int(ray_len / (self.resolution * 0.5))
            for s in range(num_steps):
                dist = s * (self.resolution * 0.5)
                px = robot_x + dist * math.cos(beam_angle)
                py = robot_y + dist * math.sin(beam_angle)
                
                idx = self.world_to_grid(px, py)
                if idx is not None:
                    row, col = idx
                    if dist < (r - self.thickness / 2.0):
                        self.log_odds[row, col] += self.l_free
            
            # Update obstacle hit point
            if is_valid_hit:
                hit_x = robot_x + r * math.cos(beam_angle)
                hit_y = robot_y + r * math.sin(beam_angle)
                idx = self.world_to_grid(hit_x, hit_y)
                if idx is not None:
                    row, col = idx
                    self.log_odds[row, col] += self.l_occ
                    
        # Clamp log odds
        np.clip(self.log_odds, self.l_min, self.l_max, out=self.log_odds)

    def get_probability_map(self) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-self.log_odds))


## 2. Synthetic Multi-Pose LiDAR Telemetry Generator

In [ ]:
def generate_synthetic_lidar_dataset(num_scans: int = 6) -> pl.DataFrame:
    records = []
    num_beams = 180
    angles = np.linspace(-math.pi, math.pi, num_beams)
    
    # Synthetic obstacle boxes [(x_min, x_max, y_min, y_max), ...]
    boxes = [
        (5.0, 8.0, 2.0, 5.0),
        (8.0, 11.0, -6.0, -3.0),
        (12.0, 14.0, -1.0, 3.0),
    ]
    
    for scan_id in range(num_scans):
        rx = scan_id * 2.5
        ry = 0.0
        rtheta = 0.0
        
        for a in angles:
            beam_dir = rtheta + a
            measured_r = 15.0 # default max range
            for x0, x1, y0, y1 in boxes:
                for step_r in np.linspace(0.5, 15.0, 150):
                    px = rx + step_r * math.cos(beam_dir)
                    py = ry + step_r * math.sin(beam_dir)
                    if x0 <= px <= x1 and y0 <= py <= y1:
                        measured_r = min(measured_r, step_r)
                        break
                        
            # Add Gaussian noise
            if measured_r < 15.0:
                measured_r += float(np.random.normal(0, 0.04))
                
            records.append({
                "scan_id": scan_id,
                "robot_x": rx,
                "robot_y": ry,
                "robot_theta": rtheta,
                "beam_angle": float(a),
                "measured_range": float(measured_r)
            })
            
    return pl.DataFrame(records)

df_lidar = generate_synthetic_lidar_dataset(num_scans=6)
print(f"Generated {len(df_lidar)} LiDAR ray measurements across 6 vehicle poses.")


## 3. Sequential Grid Updating & Interactive Visualizations

In [ ]:
grid = OccupancyGrid2D(x_min=-5.0, x_max=25.0, y_min=-15.0, y_max=15.0, resolution=0.2)

unique_scans = df_lidar["scan_id"].unique().sort().to_list()
robot_poses_x = []
robot_poses_y = []

for s_id in unique_scans:
    scan_df = df_lidar.filter(pl.col("scan_id") == s_id)
    rx = scan_df["robot_x"][0]
    ry = scan_df["robot_y"][0]
    rtheta = scan_df["robot_theta"][0]
    ranges = scan_df["measured_range"].to_numpy()
    angles = scan_df["beam_angle"].to_numpy()
    
    robot_poses_x.append(rx)
    robot_poses_y.append(ry)
    
    grid.update_scan(rx, ry, rtheta, ranges, angles)

p_map = grid.get_probability_map()

# 1. 2D Spatial Heatmap with Trajectory Overlay
x_coords = np.linspace(grid.x_min, grid.x_max, grid.width)
y_coords = np.linspace(grid.y_min, grid.y_max, grid.height)

fig_map = go.Figure()

fig_map.add_trace(go.Heatmap(
    z=p_map,
    x=x_coords,
    y=y_coords,
    colorscale="Viridis",
    zmin=0.0,
    zmax=1.0,
    colorbar=dict(title="P(Occupied)")
))

fig_map.add_trace(go.Scatter(
    x=robot_poses_x,
    y=robot_poses_y,
    mode="markers+lines",
    marker=dict(size=10, color="red", symbol="triangle-right"),
    line=dict(color="red", width=2, dash="dash"),
    name="Vehicle Trajectory"
))

# Ground truth obstacle boxes for reference
boxes = [(5.0, 8.0, 2.0, 5.0), (8.0, 11.0, -6.0, -3.0), (12.0, 14.0, -1.0, 3.0)]
for x0, x1, y0, y1 in boxes:
    fig_map.add_shape(
        type="rect",
        x0=x0, y0=y0, x1=x1, y1=y1,
        line=dict(color="crimson", width=2),
        fillcolor="rgba(255, 0, 0, 0.1)"
    )

fig_map.update_layout(
    title="2D LiDAR Occupancy Grid Probability Map with Vehicle Trajectory",
    xaxis_title="Global X (m)",
    yaxis_title="Global Y (m)",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=850,
    height=650
)
fig_map.show()


In [ ]:
# 2. 1D Cross-Section Slice & Log-Odds Transformation Curves
fig_diag = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Occupancy Probability Cross-Section at Y = 3.5 m",
        "Log-Odds to Probability Sigmoid Function"
    )
)

target_y = 3.5
row_idx = int((target_y - grid.y_min) / grid.resolution)
p_slice = p_map[row_idx, :]

fig_diag.add_trace(
    go.Scatter(x=x_coords, y=p_slice, mode="lines+markers",
               line=dict(color="#1f77b4", width=3),
               marker=dict(size=4),
               name="P(Occupied) Slice"),
    row=1, col=1
)
fig_diag.add_hline(y=0.5, line_dash="dot", line_color="gray", annotation_text="Prior P=0.5", row=1, col=1)
fig_diag.add_hline(y=0.7, line_dash="dash", line_color="red", annotation_text="Occupied Threshold", row=1, col=1)

l_vals = np.linspace(-6, 6, 200)
p_vals = 1.0 / (1.0 + np.exp(-l_vals))

fig_diag.add_trace(
    go.Scatter(x=l_vals, y=p_vals, mode="lines",
               line=dict(color="#2ca02c", width=3),
               name="P = 1 / (1 + e^-l)"),
    row=1, col=2
)
fig_diag.add_vline(x=0.0, line_dash="dot", line_color="gray", row=1, col=2)
fig_diag.add_hline(y=0.5, line_dash="dot", line_color="gray", row=1, col=2)

fig_diag.update_layout(
    title="Grid Mapping Diagnostics: Spatial Cross-Section & Probability Mapping",
    height=450,
    showlegend=False
)
fig_diag.show()


## 4. Verification & Validation Tests

In [ ]:
empty_grid = OccupancyGrid2D()
assert np.allclose(empty_grid.get_probability_map(), 0.5), "Initial probability map must be uniformly 0.5!"

occ_count = np.sum(p_map > 0.7)
free_count = np.sum(p_map < 0.3)
print(f"Mapped statistics: {occ_count} occupied cells (P > 0.7), {free_count} free cells (P < 0.3).")
assert occ_count > 50, "Obstacle cells were not properly accumulated!"
assert free_count > 200, "Free space rays were not properly carved!"

print("✅ All Occupancy Grid tests passed successfully!")
